# Functions in Python — Session 6

Welcome to Session 6. Up to now you have written *scripts* — flat sequences of statements. In this session you learn to write **functions**: named, reusable blocks of logic that take inputs, do work, and hand back a result. Functions are the single most important organisational tool in Python. Every program of more than a few lines is built out of functions, every library you import is a bundle of functions, and the standard Python functions like `len`, `print`, and `map` are the models you will imitate.

A function gives you three things at once:

1. **Reusability** — write the tricky logic once, call it a thousand times. The same block that computes loan interest today is used unchanged in next month's script.
2. **Abstraction** — callers only need to know the inputs and the output, not the internals. That separation of *what* from *how* is why huge codebases stay manageable.
3. **Modularity & readability** — a program reads top-down as a list of named actions (`load_data(...)`, `clean_data(...)`, `train_model(...)`) instead of one long wall of statements.

Here is the roadmap for this session:

1. **Anatomy of a function** — `def`, parameters, indented body, docstring, and `return`. What makes a call work.
2. **Two points of view** — the author who defines the function versus the user who only calls it.
3. **Arguments** — default, positional, and keyword arguments, and the rules for mixing them.
4. **`*args` and `**kwargs`** — collecting a variable number of positional or keyword arguments.
5. **`return` and `None`** — what a function hands back, and the special "no value" that makes bugs when ignored.
6. **Variable scope** — global vs local names, why assigning inside a function shadows the outside, and `UnboundLocalError`.
7. **Nested functions & recursion danger** — functions defined inside functions, and the crash that comes when recursion never stops.
8. **First-class citizens** — functions are objects: assignable, storable, passable, returnable.
9. **Lambda functions** — tiny anonymous one-expression functions.
10. **Higher-order functions** — `map`, `filter`, `reduce`, which take functions as arguments.

Through every topic runs one idea: **a function is a value with behaviour**. Once that clicks, everything else — passing functions around, lambdas, and `map`/`filter`/`reduce` — feels natural. Let us start by building the very first function and dissecting it line by line.


### Theory : Anatomy of a function — def, docstring, return

```python
def name_of_function(parameter1, parameter2):
    """docstring — explains what this function does"""
    # body — indented lines
    return something
```

- `def` starts the definition; the **name** follows, then `(parameters)` and a colon.
- The **body must be indented** (usually 4 spaces) — like `if`/`for`, the indentation *is* the block in Python.
- The **docstring** (triple quotes, right after the `def` line) documents the purpose, inputs, and output. It is optional but strongly recommended; `help(f)` and `f.__doc__` show it later.
- `return` hands a value back to the caller. Without a `return`, the function returns `None`.
- Later you **call** it: `name_of_function(arg1, arg2)` — the arguments fill the parameters in order.

```python
def is_even(num):
    """input - an integer ; output - 'even' / 'odd'"""
    if type(num) == int:
        return 'even' if num % 2 == 0 else 'odd'
    return 'not an integer'
```

Functions exist so you write a tricky logic ONCE and reuse it everywhere — the whole point of this session.


## 🔬 Deep Dive : Anatomy of a function

```python
def is_even(num):
    """
    This function returns if given number is even
    """
    return num % 2 == 0
```

Parts:
1. `def` — keyword
2. `is_even` — function name (identifier)
3. `(num)` — parameters (inputs)
4. `"""docstring"""` — documentation (optional)
5. body — indented lines
6. `return` — output value

- Call: `is_even(4)` — Even
- Builtin functions have __doc__ too: `print(len.__doc__)`
- Function name → first-class (object)
- Parameters vs arguments: parameters = definition side; arguments = call side
- default returns None if no return
- Docstring = help() access

In [1]:
def is_even(num):
  """
  This function returns if a given number is odd or even
  input - any valid integer
  output - odd/even
  created on - 16th Nov 2022
  """
  if type(num) == int:
    if num % 2 == 0:
      return 'even'
    else:
      return 'odd'
  else:
    return 'pagal hai kya?'

In [2]:
# function
# function_name(input)
for i in range(1,11):
  x = is_even(i)
  print(x)

odd
even
odd
even
odd
even
odd
even
odd
even


In [3]:
print(type.__doc__)

type(object) -> the object's type
type(name, bases, dict, **kwds) -> a new type


### 2 Point of views

### Theory : Two points of view - author vs user

EVERY function can be seen from two sides :

**1. The AUTHOR (definer)**
- Decides what inputs it needs (parameters) and what it computes.
- Writes the docstring so others understand it.
- Controls the return value.

**2. The USER (caller)**
- Only cares : "give me inputs, get back the result".
- Reads `type.__doc__`, `print.__doc__` to LEARN a built-in they did not write.
- Calling with a bad input should give a helpful error, not a crash.

**Parameters vs Arguments - the one-line difference :**
```python
def power(a, b):   # a, b = PARAMETERS (names, at definition time)
    return a ** b
power(2, 3)        # 2, 3 = ARGUMENTS (values, at call time)
```
Parameters are the empty boxes in the recipe; arguments are the ingredients you drop in.


## 🔬 Deep Dive : Author vs User POV

```python
is_even('hello')   # ❌ TypeError: % not supported for str and int
```

Two perspectives:
**Author** (who writes function)
- Defines parameters, logic, return
- expects types reasonable
- docstring explains contract
- Validate/type-hint inputs

**User** (who calls)
- Only needs name, inputs, output
- Doesn't need internal implementation
- Reads docstring/help
- Must pass right types — else errors

- Function = abstraction: interface (what) vs implementation (how)
- Similar to library: user calls print/math.sqrt without knowing internals
- Contracts: input type assumptions — Python dynamic, no compile check → runtime errors

In [4]:
is_even('hello')

'pagal hai kya?'

### Parameters Vs Arguments

### Types of Arguments

- Default Argument
- Positional Argument
- Keyword Argument

### Theory : Default, Positional & Keyword arguments

```python
def power(a=1, b=1):     # DEFAULT arguments
    return a ** b
```

1. **Default values** - if the caller passes nothing, the default is used :
```python
power()        # 1 (both defaults)
```

2. **Positional arguments** - matched by ORDER :
```python
power(2, 3)    # a=2, b=3 -> 8
```

3. **Keyword arguments** - matched by NAME, order free :
```python
power(b=3, a=2)   # -> 8
```

**Rules :**
- Positional first, keywords after : `power(2, b=3)` OK, `power(a=2, 3)` -> SyntaxError.
- Once you use `=`, every later argument must use `=` too.
- You may mix : `power(2, b=3)`.
- Missing a required (non-default) parameter -> `TypeError`.


In [5]:
def power(a=1,b=1):
  return a**b

In [6]:
power()

1

## 🔬 Deep Dive : Default, positional & keyword args

```python
def power(a=1, b=1):
    return a**b
power()           # default a=1,b=1 → 1
power(2,3)        # POSITIONAL → a=2,b=3 → 8
power(b=3,a=2)    # KEYWORD → 8 (order irrelevant)
```

- **Default argument** — optional; if not passed, default used
- **Positional** — order maps to parameters
- **Keyword** — name=value; order flexible
- Rules:
  - positional first, then keyword: `power(2, b=3)` ✓
  - no positional after keyword: `power(a=2, 3)` ✗
  - defaults must come after non-default params
- Defaults evaluated once at def time (immutable default warning)
- Mixed ordering is common gotcha

In [7]:
# positional argument
power(2,3)

8

In [8]:
# keyword argument
power(b=3,a=2)

8

### `*args and **kwargs`
`*args and **kwargs` are special Python keywords that are used to pass the variable length of arguments to a function

### Theory : *args & **kwargs - variable-length arguments

Sometimes you don't know how many arguments a call will pass. The stars handle that.

```python
def multiply(*kwargs):      # *args collects EXTRAS into a tuple
    product = 1
    for i in kwargs:
        product *= i
    return product
multiply(1,2,3,4,5,6,7,8,9,10,12)    # any count, no list needed!
```

```python
def display(**salman):      # **kwargs collects name=value calls into a dict
    for key, value in salman.items():
        print(key, '->', value)
display(india='delhi', srilanka='colombo')
```

**Points to remember :**
1. **Order matters** : normal arguments -> `*args` -> `**kwargs`. Nothing can follow `**kwargs`.
2. `args` / `kwargs` are only a **convention** - the notebook proves it with `*kwargs` and `**salman`. The `*` is what does the magic.
3. `*args` = tuple of positional extras; `**kwargs` = dict of keyword extras.
4. Use them for flexible APIs, wrappers, decorators.


In [9]:
# *args
# allows us to pass a variable number of non-keyword arguments to a function.

def multiply(*kwargs):
  product = 1

  for i in kwargs:
    product = product * i

  print(kwargs)
  return product

## 🔬 Deep Dive : *args & **kwargs

```python
def multiply(*args):
    result = 1
    for x in args:
        result *= x
    return result
multiply(1,2,3,4,5,6,7,8,9,10,12)
```

- `*args` — collects VARIABLE NUMBER of positional args into TUPLE
- **kwargs — collects keyword args into DICT
```python
def display(**kwargs):
    for k, v in kwargs.items():
        print(k, '->', v)
display(india='delhi', srilanka='colombo')
```

- Name `args`/`kwargs` convention (any name works)
- Order: `def f(a, b, *args, **kwargs)`
- Required before *args; **kwargs last
- Unpack inverse: `f(*L)`, `f(**d)` — spread into call
- Real use: wrappers, decorators, flexible libs
- Empty allowed: *args=(), **kwargs={}

In [10]:
multiply(1,2,3,4,5,6,7,8,9,10,12)

(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12)


43545600

In [11]:
# **kwargs
# **kwargs allows us to pass any number of keyword arguments.
# Keyword arguments mean that they contain a key-value pair, like a Python dictionary.

def display(**salman):

  for (key,value) in salman.items():
    print(key,'->',value)


In [12]:
display(india='delhi',srilanka='colombo',nepal='kathmandu',pakistan='islamabad')

india -> delhi
srilanka -> colombo
nepal -> kathmandu
pakistan -> islamabad


##### Points to remember while using `*args and **kwargs`

- order of the arguments matter(normal -> `*args` -> `**kwargs`)
- The words “args” and “kwargs” are only a convention, you can use any name of your choice


### How Functions are executed in memory?

### Without return statement

### Theory : return & None - the output channel

**A function without `return` still returns something - the special value `None` :**
```python
L = [1, 2, 3]
print(L.append(4))   # None - append changes the list but returns nothing
```
`None` means "no value". `print` shows it, and code expecting a value gets confused by it - a classic bug source.

**What return does :**
1. Ends the function immediately (code after it is skipped).
2. Hands a value back to the caller : `z = f(x)`.
3. A function may have many `return` statements (one per branch).
4. With no `return` at all, the function still finishes by returning None.

**Rules for clean functions :** if the function COMPUTES something, return it; if it does an ACTION (print, modify), returning None is normal.


## 🔬 Deep Dive : return & None

```python
def f(x):
    x = x + 1        # no return statement
print(f(5))          # None — implicit return
L = [1,2,3]
print(L.append(4))   # None — append mutates, returns None
```

- No return → function returns **None**
- `print(L.append(4))` → None (method mutates, no return)
- Side-effect functions (like append) vs pure (return)
- `return` without value → None
- None in bool context → falsy
- Implicit None — classic interview point
- Mutation vs return — know which methods behave how

In [13]:
L = [1,2,3]
print(L.append(4))
print(L)

None
[1, 2, 3, 4]


### Variable Scope

### Theory : Scope - global vs local

A name is either known everywhere (global) or only inside a function (local).

```python
x = 5                     # GLOBAL x
def g(y):
    print(x)              # reads global x - allowed
    print(x + 1)
g(x)
print(x)                  # -> 5, 6, 5
```

**Local shadows global :**
```python
x = 5
def f(y):
    x = 1                 # a LOCAL x is created - the global is untouched
    x += 1
    print(x)              # 2
f(x)
print(x)                  # 5
```

**The trap - UnboundLocalError :**
```python
x = 5
def h(y):
    x += 1                # reading x AND assigning it -> thinks x is local -> ERROR
```
Assignment ANYWHERE in the function makes Python treat the name as local, even if you only meant to read the global.

**Passing values in :** numbers/strings are passed by value - the function gets a copy (`x = x + 1` inside does NOT move the outer x). To get a result out, there is `return`.


## 🔬 Deep Dive : Scope — global vs local variables

```python
x = 5                    # GLOBAL — module level
def f(y):
    x = 1                # LOCAL — own x
    print(x)
f(10)                    # prints 1 — local wins
print(x)                 # 5 outside — global intact
```

Rules:
- Variables created inside function = **local**
- Top-level = **global** — visible inside (read), not writable:
```python
def h(y):
    x += 1      # ❌ UnboundLocalError
```
- To write global: `global x`
- Parameter y also local
- LEGB: Local → Enclosing → Global → Builtin
- Read allowed; assignment creates local
- Integers immutable — passing copies reference

In [14]:
def g(y):
    print(x)
    print(x+1)
x = 5
g(x)
print(x)

5
6
5


In [15]:
def f(y):
    x = 1
    x += 1
    print(x)
x = 5
f(x)
print(x)

2
5


In [16]:
def h(y):
    x += 1
x = 5
print(x)
h(x)

5


UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

In [17]:
def f(x):
   x = x + 1
   print('in f(x): x =', x)
   return x

x = 3
z = f(x)
print('in main program scope: z =', z)
print('in main program scope: x =', x)

in f(x): x = 4
in main program scope: z = 4
in main program scope: x = 3


### Nested Functions

### Theory : Nested functions + the recursion danger

A function defined INSIDE another function :

```python
def f():
    def g():
        print('inside g')
    g()                # f calls its inner function
    print('inside f')
```

**Nested scoping - inner vs outer names :**
```python
def g(x):
    def h():
        x = 'abc'      # h declares its OWN local x - g's x is untouched
    x = x + 1          # g's own x
    return x
```
Resolution order : local -> enclosing -> global -> built-in (LEGB).

**Recursion + the crash course :**
```python
def f():
    def g():
        print('inside g')
        f()            # f calls g, g calls f, f calls g ... forever
    g()
f()                    # eventually: RecursionError - python's safety limit
```
Every recursive call needs a **base case** that stops the chain, or recursion never ends.


## 🔬 Deep Dive : Nested functions & recursion danger

```python
def f():
    def g():
        print('inside g')
        f()        # f→g→f→g... infinite recursion
    g()
f()    # RecursionError eventually
```

- Function defined inside another = nested
- Inner can reference outer scope (closures)
- Danger: recursion without base case → RecursionError (stack depth)
- Every call pushes a stack frame (~1000 default)
- Recursion needs termination condition
- g as inner helper — encapsulation
- To modify outer var: `nonlocal`
- Nested gives scoping but over-recursion risk

In [18]:
def f():
  def g():
    print('inside function g')
    f()
  g()
  print('inside function f')

In [19]:
f()

inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside function g
inside fun

RecursionError: maximum recursion depth exceeded

In [20]:
def g(x):
    def h():
        x = 'abc'
    x = x + 1
    print('in g(x): x =', x)
    h()
    return x

x = 3
z = g(x)

in g(x): x = 4


In [21]:
def g(x):
    def h(x):
        x = x+1
        print("in h(x): x = ", x)
    x = x + 1
    print('in g(x): x = ', x)
    h(x)
    return x

x = 3
z = g(x)
print('in main program scope: x = ', x)
print('in main program scope: z = ', z)

in g(x): x =  4
in h(x): x =  5
in main program scope: x =  3
in main program scope: z =  4


### Functions are 1st class citizens

### Theory : Functions are objects - first-class citizens

In Python a function is a value, just like an int or string. Proof :

```python
type(square)      # function
id(square)        # it has an address in memory

x = square        # 1. REASSIGN to another name
x(3)              #     call it -> 9

del square        # 2. DELETE it
square(3)         #     NameError - it is gone!

L = [1, 2, square]  # 3. STORE it in containers
L[-1](9)          #     call through the list

s = {square}      #     even sets accept functions (they are hashable)
```

**Return a function from a function :**
```python
def f():
    def x(a, b):
        return a + b
    return x
f()(3, 4)         # -> 7   (f returns the inner function, then calls it)
```

**Pass a function as an argument :**
```python
def func_b(z):
    return z()    # call whatever function came in
func_b(func_a)
```
Because functions flow like any other object, we can build higher-order tools (map, filter, reduce) - next section!


In [22]:
# type and id
def square(num):
  return num**2

type(square)

id(square)

1702426338656

## 🔬 Deep Dive : Functions are first-class objects

```python
def square(num): return num**2
print(type(square))   # <class 'function'>
print(id(square))     # memory address — it's an OBJECT

x = square            # reassign — x points same function
x(3)                  # 9

L = [1,2,3,4,square] # store in list
L[-1](3)              # 9

del square            # delete name
square(3)             # NameError — deleted

def outer():
    def inner(a,b): return a+b
    return inner      # RETURNING a function
func = outer()
func(2,3)             # 5

def func_a(): print('a')
def func_b(fn):       # PASS function as argument
    fn()
func_b(func_a)
```

First-class citizen = function can be:
1. assigned to variable
2. passed as argument
3. returned from function
4. stored in data structures (list, set, dict)
5. have attributes

- This powers map/filter/reduce, decorators, callbacks
- cfg: function name = reference to object

In [23]:
# reassign
x = square
id(x)
x(3)

9

In [24]:
a = 2
b = a
b

2

In [25]:
# deleting a function
del square

In [30]:
def square(num):
  return num**2

square(3)

9

In [31]:
# storing
L = [1,2,3,4,square]
L[-1](3)

9

In [32]:
s = {square}
s

{<function __main__.square(num)>}

In [33]:
# returning a function


In [34]:
def f():
    def x(a, b):
        return a+b
    return x
    
val = f()(3,4)
print(val)

7


In [35]:
# function as argument

In [36]:
def func_a():
    print('inside func_a')

def func_b(z):
    print('inside func_c')
    return z()

print(func_b(func_a))

inside func_c
inside func_a
None


### Benefits of using a Function

- Code Modularity
- Code Readibility
- Code Reusability

### Lambda Functions

A lambda function is a **small anonymous function** — it has no name and is written in one line.

A lambda can take any number of arguments, but its body can contain only **one expression**. The value of that expression is automatically returned.

```python
lambda x: x**2            # the function itself (no name)
(lambda x: x**2)(5)       # call it straight away -> 25
```

Lambdas are most useful when passed inline to higher-order functions like `map`, `filter`, and `sorted` — you will see exactly that pattern in the upcoming cells. The next cell explains the full syntax and when to prefer a lambda over a normal `def`.


### Theory : Lambda - the small anonymous function

A lambda is a function with NO name and only ONE expression :

```python
lambda x: x**2          # the function itself
a = lambda x: x**2      # assign it a name, then
a(5)                    # -> 25

add = lambda x, y: x + y
add(5, 2)               # -> 7
```

**Lambda vs a normal def :**
| def | lambda |
| --- | ------ |
| has a name | anonymous (used on the spot) |
| many statements | ONE expression only |
| needs return | the expression IS the return value |
| reusable everywhere | usually used inline once |

**Where lambdas shine - tiny one-liners :**
```python
(lambda s: 'a' in s)('hello')          # -> False
(lambda x: 'even' if x % 2 == 0 else 'odd')(6)   # -> 'even' (ternary inside!)
```
By themselves lambdas feel weak - their power appears when passed INTO higher order functions.


## 🔬 Deep Dive : Lambda functions

```python
lambda x: x**2          # anonymous function — unused alone
a = lambda x,y: x+y
a(5,2)                  # 7
```

- `lambda args: expression` — one-liner anonymous
- Expression only: no statements, no explicit return
- Storable, passable, returnable — like def
- No name by default → inline use
- Bodies: `lambda s: 'a' in s`, ternary: `lambda x: 'even' if x%2==0 else 'odd'`
- Common with map/filter/sort
  
| | lambda | normal def |
|---|---|---|
| name | no | yes |
| body | one expression | multiple statements |
| return | implicit | explicit |
| use | simple inline | complex |

- Heavier logic → def
- Docs: lambda no docstring

In [37]:
# x -> x^2
lambda x:x**2

<function __main__.<lambda>(x)>

In [38]:
# x,y -> x+y
a = lambda x,y:x+y
a(5,2)

7

#### Diff between lambda vs Normal Function

- No name
- lambda has no return value(infact,returns a function)
- lambda is written in 1 line
- not reusable

Then why use lambda functions?<br>
**They are used with HOF**


In [39]:
# check if a string has 'a'
a = lambda s:'a' in s
a('hello')

False

In [40]:
# odd or even
a = lambda x:'even' if x%2 == 0 else 'odd'
a(6)

'even'

### Higher Order Functions

### Theory : Higher Order Functions (HOF)

A function that TAKES a function and/or RETURNS a function is a higher-order function.

```python
def square(x): return x**2
def cube(x):   return x**3

def transform(f, L):          # f is a FUNCTION parameter
    out = []
    for i in L:
        out.append(f(i))
    print(out)

transform(lambda x: x**3, [1,2,3,4,5])   # -> [1, 8, 27, 64, 125]
```
- `transform` does not care WHICH function it receives - it just applies it.
- `square` and `cube` are the same shape, so either plugs in.

**The three famous HOFs are coming :**
- `map`    - apply a function to every item.
- `filter` - keep only items passing a test.
- `reduce` - fold many items into one.

HOFs let you say WHAT to do (transform each item) without writing the loop boilerplate yourself.


## 🔬 Deep Dive : Higher Order Functions (HOF)

```python
def apply(fn, x):
    return fn(x)
apply(square, 3)      # 9 — function as ARG

def make_power(n):
    def power(x):
        return x**n
    return power      # function RETURNED
square_3 = make_power(3)
square_3(2)           # 8
```

- HOF = takes function and/or returns function
- map/filter/reduce — the canonical HOFs
- Decorators build on HOF
- callbacks, event handlers
- Functional programming cleats
- Composition functional pipelines
- Basis for the next three topics (map/filter/reduce)

In [41]:
# Example

def square(x):
  return x**2

def cube(x):
  return x**3

# HOF
def transform(f,L):
  output = []
  for i in L:
    output.append(f(i))

  print(output)

L = [1,2,3,4,5]

transform(lambda x:x**3,L)

[1, 8, 27, 64, 125]


### Map

### Theory : map(function, iterable)

`map` applies a function to EVERY item of an iterable : `map(f, L)` = `[f(x) for x in L]`.

```python
list(map(lambda x: x**2, [1,2,3,4,5]))   # -> [1, 4, 9, 16, 25]

L = [1,2,3,4,5]
list(map(lambda x: 'even' if x%2 == 0 else 'odd', L))  # labels

# map over a list of DICTS - extract one field
users = [{'name':'Rahul','gender':'male'},
         {'name':'Ankita','gender':'female'}]
list(map(lambda u: u['gender'], users))   # -> ['male', 'female']
```

**Important :** `map` returns a LAZY iterator, not a list. Use `list(map(...))` to force it (or loop it). It stops early if the iterable does.

**Perfect partner of lambda :** the function often has no reason to live beyond this one call, so an inline lambda is ideal.


## 🔬 Deep Dive : map(function, iterable)

```python
list(map(lambda x: x**2, [1,2,3,4,5]))   # [1,4,9,16,25]
list(map(lambda x: 'even' if x%2==0 else 'odd', [1,2,3,4,5]))
users = [{'name':'R'},{'name':'S'},...]
list(map(lambda u: u['name'], users))     # extract field
```

- Applies function to EVERY item
- Returns map OBJECT (iterator) — wrap in list()
- Function can be lambda or named
- Multiple iterables: map(f, a, b) → f(x,y)
- Like list comprehension — map is the functional twin
- Lazy — one pass produced on list()
- Example: `map(str, [1,2,3])` → '1','2','3'
- Use for field extraction, transforms, type conversion

In [42]:
# square the items of a list
list(map(lambda x:x**2,[1,2,3,4,5]))

[1, 4, 9, 16, 25]

In [43]:
# odd/even labelling of list items
L = [1,2,3,4,5]
list(map(lambda x:'even' if x%2 == 0 else 'odd',L))

['odd', 'even', 'odd', 'even', 'odd']

In [44]:
# fetch names from a list of dict

users = [
    {
        'name':'Rahul',
        'age':45,
        'gender':'male'
    },
    {
        'name':'Nitish',
        'age':33,
        'gender':'male'
    },
    {
        'name':'Ankita',
        'age':50,
        'gender':'female'
    }
]

list(map(lambda users:users['gender'],users))

['male', 'male', 'female']

### Filter 

### Theory : filter(function, iterable)

`filter` keeps only the items for which the function returns `True` :

```python
L = [3,4,5,6,7]
list(filter(lambda x: x > 5, L))     # -> [6, 7]

fruits = ['apple', 'guava', 'cherry']
list(filter(lambda x: x.startswith('a'), fruits))  # -> ['apple']
```

**How to read it :**
- The function is a PREDICATE - returns True/False, does not transform.
- Items that pass stay; items that fail are dropped.
- Original order is preserved.
- Returns a lazy iterator -> wrap with `list()`.

map CHANGES every item; filter SELECTS some items. They compose beautifully : `filter` first, then `map`.


## 🔬 Deep Dive : filter(function, iterable)

```python
L = [3,4,5,6,7]
list(filter(lambda x: x>5, L))   # [6,7]
fruits = ['apple','guava','cherry']
list(filter(lambda f: f.startswith('a'), fruits))  # ['apple']
```

- Keeps only items where function returns **True**
- Function must return bool/truthy
- Returns iterator — wrap list()
- `filter(None, L)` — removes falsy values
- Equivalent: `[x for x in L if cond]`
- FUNCTION filters, comprehension readable
- Doesn't transform values — only selects
- Pythonic for whitelist/keep patterns

In [45]:
# numbers greater than 5
L = [3,4,5,6,7]

list(filter(lambda x:x>5,L))

[6, 7]

In [46]:
# fetch fruits starting with 'a'
fruits = ['apple','guava','cherry']

list(filter(lambda x:x.startswith('a'),fruits))

['apple']

### Reduce

### Theory : reduce(function, iterable) - folding

`reduce` repeatedly combines items into one final value. From the functools module (not built-in like map/filter) :

```python
import functools
functools.reduce(lambda x, y: x + y, [1,2,3,4,5])   # -> 15
```

**How it works step by step :**
```python
((((1+2)+3)+4)+5)   # the lambda takes the running total + the next item
```

**Find the maximum :**
```python
functools.reduce(lambda x, y: x if x > y else y, [23, 11, 45, 10, 1])  # -> 45
```

**Notes :**
- The function takes TWO arguments (accumulator, current item).
- An initial value can be passed : `reduce(f, L, start)` (uses start instead of L[0]).
- Use it when a whole collection must "collapse" into one result (sum, product, max, join). If it feels forced, a plain loop is fine.


## 🔬 Deep Dive : reduce(function, iterable) — folding

```python
import functools
functools.reduce(lambda x,y: x+y, [1,2,3,4,5])   # sum
functools.reduce(lambda x,y: x if x>y else y, [23,11,45,10,1])  # 45
```

How reduce works:
```
reduce(f, [a,b,c,d]) = f(f(f(a,b),c),d)
```
- Repeatedly applies function to accumulated result + next item
- Single final value
- `sum()`, `max()`, `min()` are reduce special cases
- Needs `import functools` (reduced from builtins)
- Useful: aggregates like product, gcd, concat, min/max
- Readability: prefer sum/max when built-in exists
- Initializer: `reduce(f, iterable, start)`
- Folding/catamorphism functional concept

In [47]:
# sum of all item
import functools

functools.reduce(lambda x,y:x+y,[1,2,3,4,5])

15

In [48]:
# find min
functools.reduce(lambda x,y:x if x>y else y,[23,11,45,10,1])

45

## Summary

In this session you built complete functions and understood how they work under the hood. You learned the anatomy — `def`, name, parameters, indented body, docstring, and `return` — and the two points of view of an author and a user. You mastered argument passing: defaults, positional, keyword, and the variable-length `*args`/`**kwargs`. You saw that a function without `return` silently returns `None`, a frequent bug source. Scope taught you that names inside a function are local and that assigning anywhere in a function makes Python treat the name as local (hence `UnboundLocalError`); the LEGB rule (Local → Enclosing → Global → Builtin) summarises name resolution. Nested functions introduced inner helpers and the recursion danger of missing base cases. Functions proved to be first-class citizens — assignable, storable, passable, returnable — which is exactly what makes **lambdas** (tiny anonymous one-expression functions) and the **higher-order functions** `map`, `filter`, and `reduce` work. Write a few small functions each day; that repetition is what turns this anatomy into reflex.
